In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.layers import Input, Conv2D, concatenate
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import MeanAbsoluteError
from PIL import Image
import hashlib
from ecdsa import SigningKey, VerifyingKey, NIST256p, ellipticcurve
from ecdsa.curves import Curve
import binascii

In [2]:
# %% [markdown]
"""
## 1. Data Loading and Preprocessing

First, we'll implement functions to load and preprocess images from the Flickr dataset (or simulate it if not available).
"""

# %%
def load_and_preprocess_images(image_paths, target_size=(64, 64)):
    """
    Load and preprocess images from given paths
    Args:
        image_paths: List of paths to images
        target_size: Tuple (height, width) for resizing
    Returns:
        Numpy array of preprocessed images
    """
    images = []
    for path in image_paths:
        try:
            img = Image.open(path)
            img = img.convert('RGB')  # Ensure RGB format
            img = img.resize(target_size)  # Resize to target dimensions
            img_array = np.array(img) / 255.0  # Normalize to [0,1]
            images.append(img_array)
        except Exception as e:
            print(f"Error loading image {path}: {e}")
    
    return np.array(images)

# Simulate loading images (replace with actual paths)
def simulate_image_loading(num_images=1000, img_size=(64, 64, 3)):
    """
    Simulate loading images by generating random noise images
    Args:
        num_images: Number of images to generate
        img_size: Tuple (height, width, channels) for image dimensions
    Returns:
        Numpy array of simulated images
    """
    return np.random.rand(num_images, *img_size)

# Example usage
# cover_images = load_and_preprocess_images(cover_image_paths)
# secret_images = load_and_preprocess_images(secret_image_paths)

# For this demo, we'll use simulated data
cover_images = simulate_image_loading(1000)
secret_images = simulate_image_loading(1000)

print(f"Cover images shape: {cover_images.shape}")
print(f"Secret images shape: {secret_images.shape}")

Cover images shape: (1000, 64, 64, 3)
Secret images shape: (1000, 64, 64, 3)


In [ ]:
# %% [markdown]
"""
## 2. Elliptic Curve Cryptography (ECC) Implementation

We'll implement ECC encryption/decryption using the 'brainpoolP256r1' curve as specified in the paper.
"""

# %%
class ECCSteganography:
    def __init__(self, curve_name='brainpoolP256r1'):
        """
        Initialize ECC with specified curve
        Args:
            curve_name: Name of the elliptic curve to use
        """
        # For this implementation, we'll use NIST256p as brainpoolP256r1 isn't directly available
        # In a production system, you would implement the exact curve from the paper
        self.curve = NIST256p
        
    def generate_keys(self):
        """
        Generate ECC key pair
        Returns:
            private_key: Private key for decryption
            public_key: Public key for encryption
        """
        private_key = SigningKey.generate(curve=self.curve)
        public_key = private_key.get_verifying_key()
        return private_key, public_key
    
    def encrypt(self, public_key, message):
        """
        Encrypt a message using ECC public key
        Args:
            public_key: Public key for encryption
            message: Message to encrypt (string or bytes)
        Returns:
            encrypted_message: Encrypted message
        """
        if isinstance(message, str):
            message = message.encode()
            
        # In a real implementation, we would use ECIES (Elliptic Curve Integrated Encryption Scheme)
        # For this demo, we'll simulate encryption by XORing with a derived key
        # Note: This is a simplified approach - use proper ECIES in production
        
        # Derive a shared secret (in real ECIES this would be done properly)
        ephemeral_private = SigningKey.generate(curve=self.curve)
        ephemeral_public = ephemeral_private.get_verifying_key()
        
        # Simulate shared secret derivation
        shared_secret = self._derive_shared_secret(ephemeral_private, public_key)
        
        # Use first 32 bytes of SHA512 of shared secret as key
        key = hashlib.sha512(shared_secret).digest()[:32]
        
        # Simple XOR encryption (in real implementation use AES)
        encrypted = bytes([m ^ k for m, k in zip(message, (key * (len(message)//32 + 1)))])
        
        # Return ephemeral public key + encrypted message
        return ephemeral_public.to_string() + encrypted
    
    def decrypt(self, private_key, encrypted_message):
        """
        Decrypt a message using ECC private key
        Args:
            private_key: Private key for decryption
            encrypted_message: Encrypted message (bytes)
        Returns:
            decrypted_message: Decrypted message
        """
        # Extract ephemeral public key (first part of message)
        pubkey_len = len(private_key.get_verifying_key().to_string())
        ephemeral_pubkey_str = encrypted_message[:pubkey_len]
        ciphertext = encrypted_message[pubkey_len:]
        
        # Reconstruct ephemeral public key
        ephemeral_pubkey = VerifyingKey.from_string(
            ephemeral_pubkey_str, 
            curve=self.curve
        )
        
        # Derive shared secret
        shared_secret = self._derive_shared_secret(private_key, ephemeral_pubkey)
        
        # Use first 32 bytes of SHA512 of shared secret as key
        key = hashlib.sha512(shared_secret).digest()[:32]
        
        # Simple XOR decryption (in real implementation use AES)
        decrypted = bytes([c ^ k for c, k in zip(ciphertext, (key * (len(ciphertext)//32 + 1)))])
        
        return decrypted
    
    def _derive_shared_secret(self, private_key, public_key):
        """Derive shared secret using ECDH"""
        # Get the point on the curve from public key
        point = public_key.pubkey.point
        
        # Multiply by private key to get shared secret point
        shared_point = private_key.privkey.secret_multiplier * point
        
        # Return x coordinate as bytes
        return shared_point.x().to_bytes(32, 'big')
    
    def text_to_image(self, text, image_size=(64, 64)):
        """
        Convert text to an image representation
        Args:
            text: Text to convert
            image_size: Target image dimensions (height, width)
        Returns:
            Numpy array representing the image
        """
        if isinstance(text, str):
            text = text.encode()
        
        # Pad text to fill image
        pixels_needed = image_size[0] * image_size[1] * 3  # RGB channels
        padded_text = text + b'\0' * (pixels_needed - len(text))
        
        # Convert to numpy array and reshape
        arr = np.frombuffer(padded_text[:pixels_needed], dtype=np.uint8)
        arr = arr.reshape(image_size[0], image_size[1], 3)
        
        return arr / 255.0  # Normalize to [0,1]
    
    def image_to_text(self, image):
        """
        Convert image back to text
        Args:
            image: Numpy array representing the image
        Returns:
            Original text
        """
        # Scale back to 0-255 and convert to bytes
        arr = (image * 255).astype(np.uint8)
        text_bytes = arr.tobytes()
        
        # Remove padding null bytes
        text_bytes = text_bytes.rstrip(b'\0')
        
        return text_bytes

# Example usage
ecc = ECCSteganography()
private_key, public_key = ecc.generate_keys()

# Encrypt and convert to image
secret_text = "This is a secret message!"
encrypted_text = ecc.encrypt(public_key, secret_text)
secret_image = ecc.text_to_image(encrypted_text)

# Convert back and decrypt
recovered_text = ecc.image_to_text(secret_image)
decrypted_text = ecc.decrypt(private_key, recovered_text).decode()

print(f"Original text: {secret_text}")
print(f"Decrypted text: {decrypted_text}")

In [ ]:
# %% [markdown]
"""
## 3. Stacked Autoencoder Model Architecture

Now we'll implement the Stacked Autoencoder model with:
- Preparation Network (2 convolutional layers)
- Hiding Network (5 convolutional layers)
- Reveal Network (5 convolutional layers)
"""

# %%
from tensorflow.keras import Input, Model
from tensorflow.keras.layers import Conv2D, concatenate
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import MeanAbsoluteError

def build_preparation_network(input_shape=(64, 64, 3)):
    """
    Build the Preparation Network (2 conv layers)
    Args:
        input_shape: Shape of input images
    Returns:
        Keras Model
    """
    inputs = Input(shape=input_shape)

    # First convolutional layer
    x = Conv2D(65, (3, 3), activation='relu', padding='same')(inputs)

    # Second convolutional layer
    x = Conv2D(65, (3, 3), activation='relu', padding='same')(x)

    return Model(inputs, x, name='PreparationNetwork')

def build_hiding_network(cover_shape=(64, 64, 3), secret_shape=(64, 64, 65)):
    """
    Build the Hiding Network (5 conv layers)
    Args:
        input_shape: Shape of input images (cover + prepared secret)
    Returns:
        Keras Model
    """
    # Two inputs - cover image and prepared secret
    cover_input = Input(shape=cover_shape, name='cover_input')
    secret_input = Input(shape=secret_shape, name='secret_input')

    # Concatenate along channel dimension
    x = concatenate([cover_input, secret_input], axis=-1)

    # Five convolutional layers
    for _ in range(5):
        x = Conv2D(65, (3, 3), activation='relu', padding='same')(x)

    # Final layer to get back to 3 channels
    output = Conv2D(3, (3, 3), activation='sigmoid', padding='same')(x)

    return Model([cover_input, secret_input], output, name='HidingNetwork')

def build_reveal_network(input_shape=(64, 64, 3)):
    """
    Build the Reveal Network (5 conv layers)
    Args:
        input_shape: Shape of input images (container)
    Returns:
        Keras Model
    """
    inputs = Input(shape=input_shape)

    # Five convolutional layers
    x = inputs
    for _ in range(5):
        x = Conv2D(65, (3, 3), activation='relu', padding='same')(x)

    # Final layer to reconstruct secret
    output = Conv2D(3, (3, 3), activation='sigmoid', padding='same')(x)

    return Model(inputs, output, name='RevealNetwork')

def build_stacked_autoencoder(input_shape=(64, 64, 3)):
    """
    Build the complete Stacked Autoencoder model
    Args:
        input_shape: Shape of input images
    Returns:
        encoder_model: Model that takes cover and secret, outputs container
        decoder_model: Model that takes container, outputs secret
        full_model: Combined model for training
    """
    # Build the three networks
    prep_net = build_preparation_network(input_shape)
    hide_net = build_hiding_network(input_shape, (input_shape[0], input_shape[1], 65)) # Sesuaikan shape secret
    reveal_net = build_reveal_network(input_shape)

    # Encoder model (cover + secret -> container)
    cover_input = Input(shape=input_shape, name='cover_input')
    secret_input = Input(shape=input_shape, name='secret_input')

    prepared_secret = prep_net(secret_input)
    container_output = hide_net([cover_input, prepared_secret])

    encoder_model = Model(
        inputs=[cover_input, secret_input],
        outputs=container_output,
        name='Encoder'
    )

    # Decoder model (container -> secret)
    container_input = Input(shape=input_shape, name='container_input')
    revealed_secret = reveal_net(container_input)

    decoder_model = Model(
        inputs=container_input,
        outputs=revealed_secret,
        name='Decoder'
    )

    # Full model for training (cover + secret -> container -> secret)
    full_model_input_cover = Input(shape=input_shape, name='cover_input')
    full_model_input_secret = Input(shape=input_shape, name='secret_input')
    prepared_secret_output = prep_net(full_model_input_secret)
    container_output_full = hide_net([full_model_input_cover, prepared_secret_output])
    revealed_output_full = reveal_net(container_output_full)

    full_model = Model(
        inputs=[full_model_input_cover, full_model_input_secret],
        outputs=[container_output_full, revealed_output_full],
        name='FullModel'
    )

    return encoder_model, decoder_model, full_model

# Build the model
encoder, decoder, full_model = build_stacked_autoencoder()

# Compile the model with Adam optimizer and custom loss
def custom_loss(y_true, y_pred):
    """Custom loss function as defined in the paper"""
    return MeanAbsoluteError()(y_true, y_pred)

full_model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss={
        'FullModel': custom_loss, # Menggunakan nama model full
        'RevealNetwork': custom_loss # Target untuk reveal network adalah output kedua
    },
    loss_weights=[1.0, 1.0] # β=1 as in the paper
)

# Display model summary
full_model.summary()

In [ ]:
# %% [markdown]
"""
## 4. Training the Stacked Autoencoder

Now we'll train the model using the simulated data.
"""

# %%
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import MeanAbsoluteError

# Asumsi: cover_images dan secret_images sudah dibuat sebelumnya
# Membuat data dummy jika belum ada
np.random.seed(42)
cover_images = np.random.rand(100, 64, 64, 3).astype('float32')
secret_images = np.random.rand(100, 64, 64, 3).astype('float32')

def train_model(model, cover_images, secret_images, epochs=10, batch_size=32):
    """
    Train the Stacked Autoencoder model
    Args:
        model: The full model to train
        cover_images: Array of cover images
        secret_images: Array of secret images
        epochs: Number of training epochs
        batch_size: Batch size for training
    Returns:
        Training history
    """
    # The target outputs are the cover images (for container) and secret images (for revealed secret)
    history = model.fit(
        [cover_images, secret_images],
        [cover_images, secret_images],  # Target kedua output dari full_model
        epochs=epochs,
        batch_size=batch_size,
        validation_split=0.2,
        shuffle=True
    )

    return history

# Build the model (pastikan ini sudah dijalankan sebelum bagian ini)
from tensorflow.keras import Input, Model
from tensorflow.keras.layers import Conv2D, concatenate

def build_preparation_network(input_shape=(64, 64, 3)):
    inputs = Input(shape=input_shape)
    x = Conv2D(65, (3, 3), activation='relu', padding='same')(inputs)
    x = Conv2D(65, (3, 3), activation='relu', padding='same')(x)
    return Model(inputs, x, name='PreparationNetwork')

def build_hiding_network(cover_shape=(64, 64, 3), secret_shape=(64, 64, 65)):
    cover_input = Input(shape=cover_shape, name='cover_input')
    secret_input = Input(shape=secret_shape, name='secret_input')
    x = concatenate([cover_input, secret_input], axis=-1)
    for _ in range(5):
        x = Conv2D(65, (3, 3), activation='relu', padding='same')(x)
    output = Conv2D(3, (3, 3), activation='sigmoid', padding='same')(x)
    return Model([cover_input, secret_input], output, name='HidingNetwork')

def build_reveal_network(input_shape=(64, 64, 3)):
    inputs = Input(shape=input_shape)
    x = inputs
    for _ in range(5):
        x = Conv2D(65, (3, 3), activation='relu', padding='same')(x)
    output = Conv2D(3, (3, 3), activation='sigmoid', padding='same')(x)
    return Model(inputs, output, name='RevealNetwork')

def build_stacked_autoencoder(input_shape=(64, 64, 3)):
    prep_net = build_preparation_network(input_shape)
    hide_net = build_hiding_network(input_shape, (input_shape[0], input_shape[1], 65))
    reveal_net = build_reveal_network(input_shape)

    cover_input = Input(shape=input_shape, name='cover_input')
    secret_input = Input(shape=input_shape, name='secret_input')
    prepared_secret = prep_net(secret_input)
    container_output = hide_net([cover_input, prepared_secret])
    encoder_model = Model(inputs=[cover_input, secret_input], outputs=container_output, name='Encoder')

    container_input = Input(shape=input_shape, name='container_input')
    revealed_secret = reveal_net(container_input)
    decoder_model = Model(inputs=container_input, outputs=revealed_secret, name='Decoder')

    full_model_input_cover = Input(shape=input_shape, name='cover_input')
    full_model_input_secret = Input(shape=input_shape, name='secret_input')
    prepared_secret_output = prep_net(full_model_input_secret)
    container_output_full = hide_net([full_model_input_cover, prepared_secret_output])
    revealed_output_full = reveal_net(container_output_full)

    full_model = Model(inputs=[full_model_input_cover, full_model_input_secret], outputs=[container_output_full, revealed_output_full], name='FullModel')
    return encoder_model, decoder_model, full_model

encoder, decoder, full_model = build_stacked_autoencoder()

def custom_loss(y_true, y_pred):
    return MeanAbsoluteError()(y_true, y_pred)

full_model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss=[custom_loss, custom_loss], # Loss untuk setiap output
    loss_weights=[1.0, 1.0]
)

# Train the model (menggunakan smaller epochs untuk demo)
history = train_model(full_model, cover_images, secret_images, epochs=500)

# Plot training history
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(history.history['HidingNetwork_loss'], label='Container Loss') # Perhatikan perubahan nama loss
plt.plot(history.history['val_HidingNetwork_loss'], label='Val Container Loss') # Perhatikan perubahan nama loss
plt.title('Container Image Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['RevealNetwork_loss'], label='Secret Loss') # Perhatikan perubahan nama loss
plt.plot(history.history['val_RevealNetwork_loss'], label='Val Secret Loss') # Perhatikan perubahan nama loss
plt.title('Secret Image Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()